# Gold Fact: TfL Line Status

Build the historical Tube line-status fact table.

This notebook:

1. Reads Silver line-status observations.
2. Resolves the correct historical `dim_line` version.
3. Resolves the London-local calendar date.
4. Creates a deterministic fact key.
5. Derives service-state indicators.
6. Validates fact grain and referential integrity.
7. Writes observations using an idempotent Delta merge.

**Source:** `workspace.urbanpulse_silver.tfl_line_status`

**Target:** `workspace.urbanpulse_gold.fact_line_status`

**Grain:** One Tube line status observation per API snapshot.

## 1. Initialise project paths

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print(f"Project root: {PROJECT_ROOT}")

## 2. Import fact-building components

In [0]:
from pyspark.sql import functions as F

from urbanpulse.transformations.fact_line_status import (
    build_fact_line_status,
)

from urbanpulse.quality.fact_line_status import (
    invalid_fact_line_status,
)

from urbanpulse.utils.delta import (
    merge_insert_only,
)

## 3. Define source and target tables

In [0]:
SILVER_TABLE = (
    "workspace."
    "urbanpulse_silver."
    "tfl_line_status"
)

DIM_LINE = (
    "workspace."
    "urbanpulse_gold."
    "dim_line"
)

DIM_DATE = (
    "workspace."
    "urbanpulse_gold."
    "dim_date"
)

TARGET_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "fact_line_status"
)

## 4. Read Silver observations and Gold dimensions

In [0]:
silver_df = spark.table(
    SILVER_TABLE
)

dim_line_df = spark.table(
    DIM_LINE
)

dim_date_df = spark.table(
    DIM_DATE
)

source_count = silver_df.count()

print(
    f"Silver observations: "
    f"{source_count}"
)

## 5. Validate Silver observation grain

Silver line-status observations must remain unique by:

`request_id + line_id + status_id`

In [0]:
duplicate_source_df = (
    silver_df
    .groupBy(
        "request_id",
        "line_id",
        "status_id",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_source_count = (
    duplicate_source_df.count()
)

if duplicate_source_count > 0:
    display(
        duplicate_source_df
    )

    raise ValueError(
        "Duplicate Silver line-status "
        "observations detected."
    )

print(
    "Silver fact grain validation passed."
)

## 6. Build fact records

Resolve each Silver observation to:

- its historical line dimension version
- its London-local date dimension row

In [0]:
fact_df = build_fact_line_status(
    silver_df=silver_df,
    dim_line_df=dim_line_df,
    dim_date_df=dim_date_df,
)

fact_count = fact_df.count()

print(
    f"Fact candidates: "
    f"{fact_count}"
)

display(
    fact_df
    .orderBy(
        F.col("snapshot_at").desc(),
        "line_id",
    )
)

## 7. Validate dimensional resolution

Every Silver observation must resolve to exactly one line dimension version and one date dimension row.

No source observations may be lost or multiplied.

In [0]:
print(
    f"Silver rows: {source_count}"
)

print(
    f"Fact rows:   {fact_count}"
)

if fact_count != source_count:
    raise ValueError(
        "Fact row count does not match "
        "Silver source row count. "
        "Investigate dimension resolution."
    )

print(
    "All source observations resolved "
    "exactly once."
)

## 8. Apply fact data-quality checks

Fact records require valid dimension keys, timestamps, source identifiers, and service-severity values.

In [0]:
invalid_df = (
    invalid_fact_line_status(
        fact_df
    )
)

invalid_count = (
    invalid_df.count()
)

print(
    f"Invalid facts: "
    f"{invalid_count}"
)

if invalid_count > 0:
    display(invalid_df)

    raise ValueError(
        f"{invalid_count} invalid "
        "line-status facts detected."
    )

print(
    "Fact quality checks passed."
)

## 9. Validate derived service indicators

Each observation must be classified as either good service or disrupted, never both.

In [0]:
invalid_flags_df = (
    fact_df
    .filter(
        F.col("is_good_service")
        ==
        F.col("is_disrupted")
    )
)

invalid_flag_count = (
    invalid_flags_df.count()
)

if invalid_flag_count > 0:
    display(
        invalid_flags_df
    )

    raise ValueError(
        "Invalid service-state "
        "classification detected."
    )

print(
    "Service-state validation passed."
)

## 10. Validate fact keys

`line_status_key` must uniquely identify every fact observation.

In [0]:
duplicate_fact_keys_df = (
    fact_df
    .groupBy(
        "line_status_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_fact_keys = (
    duplicate_fact_keys_df.count()
)

if duplicate_fact_keys > 0:
    display(
        duplicate_fact_keys_df
    )

    raise ValueError(
        "Duplicate line_status_key "
        "values detected."
    )

print(
    "Fact keys are unique."
)

## 11. Add Gold processing metadata

In [0]:
gold_df = (
    fact_df
    .withColumn(
        "created_at",
        F.current_timestamp(),
    )
)

## 12. Merge line-status facts into Gold

The fact table is insert-only.

Previously processed observations are not rewritten when the notebook is rerun.

In [0]:
merge_result = merge_insert_only(
    spark=spark,
    source_df=gold_df,
    target_table=TARGET_TABLE,
    merge_condition="""
        target.line_status_key
        =
        source.line_status_key
    """,
)

print(
    f"Gold fact table: "
    f"{merge_result}"
)

## 13. Verify line-status facts

In [0]:
%sql
SELECT
    line_status_key,
    line_key,
    line_id,
    snapshot_date_key,
    snapshot_at,
    status_severity,
    status_description,
    status_reason,
    is_good_service,
    is_disrupted
FROM workspace.urbanpulse_gold.fact_line_status
ORDER BY snapshot_at DESC, line_id;

## 14. Verify dimensional joins

In [0]:
%sql
SELECT
    d.calendar_date,
    d.day_name,
    d.is_weekend,
    d.is_bank_holiday,
    l.line_name,
    f.status_description,
    f.is_disrupted,
    f.snapshot_at
FROM workspace.urbanpulse_gold.fact_line_status f

INNER JOIN workspace.urbanpulse_gold.dim_line l
    ON f.line_key = l.line_key

INNER JOIN workspace.urbanpulse_gold.dim_date d
    ON f.snapshot_date_key = d.date_key

ORDER BY
    f.snapshot_at DESC,
    l.line_name;

In [0]:
%sql
SELECT f.*
FROM workspace.urbanpulse_gold.fact_line_status f

LEFT ANTI JOIN workspace.urbanpulse_gold.dim_line l
    ON f.line_key = l.line_key;

In [0]:
%sql
SELECT f.*
FROM workspace.urbanpulse_gold.fact_line_status f

LEFT ANTI JOIN workspace.urbanpulse_gold.dim_date d
    ON f.snapshot_date_key = d.date_key;

In [0]:
%sql
-- Verify SCD Temporal integrity - expect 0 rows
SELECT
    f.line_status_key,
    f.line_id,
    f.snapshot_at,
    l.effective_from,
    l.effective_to
FROM workspace.urbanpulse_gold.fact_line_status f

INNER JOIN workspace.urbanpulse_gold.dim_line l
    ON f.line_key = l.line_key

WHERE
    f.snapshot_at < l.effective_from

    OR (
        l.effective_to IS NOT NULL
        AND f.snapshot_at >= l.effective_to
    );

In [0]:
%sql
SELECT
    line_status_key,
    COUNT(*) AS records
FROM workspace.urbanpulse_gold.fact_line_status
GROUP BY line_status_key
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT
    status_severity,
    status_description,
    COUNT(*) AS observations
FROM workspace.urbanpulse_gold.fact_line_status
GROUP BY
    status_severity,
    status_description
ORDER BY
    status_severity,
    status_description;

In [0]:
%sql
SELECT
    l.line_name,
    f.status_description,
    f.status_reason,
    f.snapshot_at
FROM workspace.urbanpulse_gold.fact_line_status f

INNER JOIN workspace.urbanpulse_gold.dim_line l
    ON f.line_key = l.line_key

WHERE f.is_disrupted = TRUE

ORDER BY f.snapshot_at DESC;

In [0]:
%sql
SELECT
    l.line_name,

    COUNT(*) AS observations,

    SUM(
        CASE
            WHEN f.is_disrupted THEN 1
            ELSE 0
        END
    ) AS disrupted_observations,

    ROUND(
        100.0
        *
        SUM(
            CASE
                WHEN f.is_disrupted THEN 1
                ELSE 0
            END
        )
        /
        COUNT(*),
        2
    ) AS disruption_rate_pct

FROM workspace.urbanpulse_gold.fact_line_status f

INNER JOIN workspace.urbanpulse_gold.dim_line l
    ON f.line_key = l.line_key

GROUP BY
    l.line_key,
    l.line_name

ORDER BY
    disruption_rate_pct DESC,
    l.line_name;

In [0]:
%sql
SELECT
    f.snapshot_at,
    f.snapshot_date_key,
    d.calendar_date
FROM workspace.urbanpulse_gold.fact_line_status f

INNER JOIN workspace.urbanpulse_gold.dim_date d
    ON f.snapshot_date_key = d.date_key

ORDER BY f.snapshot_at DESC
LIMIT 20;

In [0]:
%sql
SELECT COUNT(*) AS facts
FROM workspace.urbanpulse_gold.fact_line_status;